# Retail Demand Lab: TimesFM 3.0

An executed analysis of real Favorita observations and saved official TimesFM 3.0 predictions. The forecasting run is separate from these lightweight analysis cells. See README for the full inference commands.

**Business question:** Do multivariate sales and extra inputs improve forecasts and inventory decisions?

**Scope:** Store 1, Quito; four families; twelve held-out weekly origins in 2017; 14-day forecasts. Future promotions are an assumed-known scenario, and prices are absent.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Portable whether opened in the project folder or its parent.
root = Path.cwd()
if not (root / 'data' / 'retail_panel.csv').exists():
    if (root / 'retail-demand-lab' / 'data').exists():
        root = root / 'retail-demand-lab'
    elif 'PROJECT_ROOT' in globals():
        root = PROJECT_ROOT
    else:
        raise FileNotFoundError('Open this notebook from the project directory.')
panel = pd.read_csv(root / 'data' / 'retail_panel.csv', parse_dates=['date'])
forecasts = pd.read_csv(root / 'results' / 'forecasts.csv')
scores = pd.read_csv(root / 'results' / 'forecast_metrics.csv')
inventory = pd.read_csv(root / 'results' / 'inventory_metrics.csv')
print('Panel rows:', len(panel), '| Observed sales:', panel.sales.notna().sum())
print('Saved forecast rows:', len(forecasts))
print(panel.groupby('family').sales.agg(['count', 'mean', 'std']).round(2).to_string())

Panel rows: 3832 | Observed sales: 3824
Saved forecast rows: 8064
              count     mean     std
family                              
BEVERAGES       956  1900.08  673.30
BREAD/BAKERY    956   373.97  113.39
DAIRY           956   707.43  196.95
GROCERY I       956  2433.41  826.48


## 1. Input and split audit

Targets are observed product-family sales, which can be fractional. Past-only traffic is measured by transaction counts. Only calendar and assumed-planned promotion inputs extend into the forecast horizon. Future sales and realized traffic never enter input features. Missing Christmas history is filled causally from the previous same weekday, while missing source targets remain identifiable in the exported panel.

In [2]:
config = json.loads((root / 'results' / 'run_config.json').read_text())
cal_last = pd.Timestamp(config['calibration_origins'][-1]) + pd.Timedelta(days=13)
test_first = pd.Timestamp(config['test_origins'][0])
assert cal_last < test_first
print('Last calibration target:', cal_last.date())
print('First test forecast:', test_first.date())
print('Context:', config['context'], '| Horizon:', config['horizon'])
print('Missing raw sales observations:', panel.sales.isna().sum())
print(config['future_input_assumption'])

Last calibration target: 2017-05-16
First test forecast: 2017-05-17
Context: 256 | Horizon: 14
Missing raw sales observations: 8
Calendar known; promotion counts assumed planned; no future traffic


## 2. Forecast comparison

WAPE = total absolute error / total observed sales. Macro MASE averages scaled errors across families. Forecast windows overlap; a separate next-week score avoids repeated dates. TimesFM uses pretrained weights without task-specific fitting; gradient boosting is a fitted conventional comparator.

In [3]:
print(scores[['model', 'wape', 'macro_mase', 'coverage80', 'pinball']].round(4).to_string(index=False))
best_timesfm = scores[scores.model.str.startswith('TimesFM')].sort_values('wape').iloc[0]
naive = scores[scores.model == 'Seasonal naive'].iloc[0]
print(f"\nBest TimesFM relative WAPE reduction vs naive: {1-best_timesfm.wape/naive.wape:.1%}")
print('\nNon-overlapping next-week scores:')
print(pd.read_csv(root / 'results' / 'next_week_metrics.csv')[['model', 'wape']].round(4).to_string(index=False))

               model   wape  macro_mase  coverage80  pinball
  TimesFM + calendar 0.0968      0.5665      0.8780  56.7825
   Gradient boosting 0.0969      0.5925      0.8616  56.6721
TimesFM + all inputs 0.0999      0.5911      0.8512  58.9816
TimesFM multivariate 0.1016      0.5928      0.8869  60.1025
  TimesFM univariate 0.1020      0.5892      0.8914  60.6554
      Seasonal naive 0.1256      0.7250      0.9226  77.5581

Best TimesFM relative WAPE reduction vs naive: 22.9%

Non-overlapping next-week scores:
               model   wape
  TimesFM + calendar 0.0910
   Gradient boosting 0.0918
TimesFM multivariate 0.0949
  TimesFM univariate 0.0953
TimesFM + all inputs 0.0965
      Seasonal naive 0.1202


![Forecast and inventory comparison](results/figures/results_overview.png)

Calendar-aware TimesFM and gradient boosting have very close observed overall scores; no statistical equivalence test was conducted. Gradient boosting receives promotion and transaction inputs absent from calendar-only TimesFM. Additional promotion and historical transaction channels did not improve the calendar-only model here. These fixed ablations cannot identify the separate causal effect of promotions or traffic.

## 3. Inspect a saved held-out forecast

These values are actual saved outputs from the official checkpoint, not generated example numbers. The offline report adds selectors for family, origin, and configuration.

In [4]:
sample = forecasts[(forecasts.model == 'TimesFM + calendar') &
                   (forecasts.origin == '2017-08-02') &
                   (forecasts.family == 'BEVERAGES')]
print(sample[['date','actual','prediction','q10','q50','q90']].round(2).to_string(index=False))
qcols = [f'q{q:02d}' for q in range(10,100,10)]
assert (np.diff(forecasts[qcols].to_numpy(), axis=1) >= -1e-5).all()
print('\nAll saved quantiles are ordered.')

      date  actual  prediction     q10     q50     q90
2017-08-02  2645.0     2496.25 2200.84 2496.25 2771.90
2017-08-03  2037.0     2206.54 1898.20 2206.54 2527.08
2017-08-04  2479.0     2481.50 2170.29 2481.50 2804.73
2017-08-05  2093.0     2529.47 2204.33 2529.47 2872.14
2017-08-06   968.0     1171.03  760.47 1171.03 1598.86
2017-08-07  2086.0     2354.81 1996.46 2354.81 2717.43
2017-08-08  2418.0     2476.26 2118.70 2476.26 2817.71
2017-08-09  2311.0     2452.01 2074.17 2452.01 2797.42
2017-08-10  2377.0     2130.26 1721.74 2130.26 2509.97
2017-08-11  1006.0     2119.84 1594.72 2119.84 2600.99
2017-08-12  1659.0     2380.86 1966.32 2380.86 2752.42
2017-08-13   803.0     1086.53  641.68 1086.53 1501.04
2017-08-14  2201.0     2230.78 1804.00 2230.78 2608.65
2017-08-15  1942.0     2411.25 2014.92 2411.25 2772.92

All saved quantiles are ordered.


![Four-family held-out forecast](results/figures/held_out_forecast.png)

## 4. Inventory policy and aggregate uncertainty

Weekly review, seven-day delivery lead time, 14-day protection period. Safety buffers use the 90th empirical percentile of signed aggregate 14-day point forecast errors from pre-test calibration. TimesFM point forecasts are medians. The procedure never sums daily 90th percentiles. Calibration windows overlap and the sample is small, so no formal 90% service guarantee is claimed.

In [5]:
buffers = pd.read_csv(root / 'results' / 'aggregate_error_buffers.csv')
print(buffers.round(2).to_string(index=False))
print('\nInventory replay:')
print(inventory[['model','fill_rate','lost_sales','average_inventory','total_cost']].round(3).to_string(index=False))
inv_best = inventory.sort_values('total_cost').iloc[0]
inv_naive = inventory[inventory.model == 'Seasonal naive'].iloc[0]
print(f"\nLowest replay cost: {inv_best.model}; reduction vs naive: {1-inv_best.total_cost/inv_naive.total_cost:.1%}")

               model       family  calibration_origins  buffer
   Gradient boosting    BEVERAGES                   12 2395.06
   Gradient boosting BREAD/BAKERY                   12  475.99
   Gradient boosting        DAIRY                   12  626.37
   Gradient boosting    GROCERY I                   12 4033.58
      Seasonal naive    BEVERAGES                   12 1796.00
      Seasonal naive BREAD/BAKERY                   12  634.58
      Seasonal naive        DAIRY                   12 1136.00
      Seasonal naive    GROCERY I                   12 4063.00
TimesFM + all inputs    BEVERAGES                   12 2383.45
TimesFM + all inputs BREAD/BAKERY                   12  339.78
TimesFM + all inputs        DAIRY                   12  812.52
TimesFM + all inputs    GROCERY I                   12 2535.41
  TimesFM + calendar    BEVERAGES                   12 2139.21
  TimesFM + calendar BREAD/BAKERY                   12  179.60
  TimesFM + calendar        DAIRY                   12 

## 5. Verify an inventory balance

For each family-day: previous on-hand + received stock − served sales = ending stock. All models use identical initial stock, calculated from the week before testing. Costs are illustrative units and observed sales are an imperfect demand proxy.

In [6]:
daily = pd.read_csv(root / 'results' / 'inventory_daily.csv')
truth = panel.pivot(index='date', columns='family', values='sales')
for (_, family), group in daily.groupby(['model','family']):
    initial = 2 * truth.loc['2017-05-10':'2017-05-16', family].sum()
    previous = np.r_[initial, group.ending_inventory.to_numpy()[:-1]]
    np.testing.assert_allclose(previous + group.arrival - group.served, group.ending_inventory, atol=1e-7)
    np.testing.assert_allclose(group.served + group.lost, group.demand, atol=1e-7)
print('Daily stock and demand balances verified for all 6 models and 4 families.')
print('\nCost sensitivity (same orders, different unit costs):')
sensitivity = pd.read_csv(root / 'results' / 'inventory_cost_sensitivity.csv')
print(sensitivity.sort_values('total_cost').groupby(['holding_rate','shortage_rate']).first()[['model','total_cost']].round(2).to_string())

Daily stock and demand balances verified for all 6 models and 4 families.

Cost sensitivity (same orders, different unit costs):
                                           model  total_cost
holding_rate shortage_rate                                  
0.005        1              TimesFM multivariate    12819.15
0.010        1              TimesFM multivariate    24304.95
             2              TimesFM multivariate    25638.31
0.050        1                TimesFM + calendar   112350.79


## Interpretation and limits

- The model with the lowest forecast error was not the model with the lowest simulated inventory cost.
- This is a single-store historical benchmark. No statistical significance, universal superiority, or actual business savings claim is justified.
- Promotion availability is assumed; target sales may be censored by stockouts; product-family units differ; no prices are included.
- Public data may overlap foundation-model pretraining, whose full lineage cannot be independently audited here.
- The inventory model omits capacity, expiry, buying costs, and terminal valuation. It supplies a reproducible experiment, not a production policy.
- Downloadable TimesFM 3.0 weights have a non-commercial, non-production license. Original data terms also apply.

For exact sources, settings, full experiment rerun commands, and six integrity tests, see the README and CASE_STUDY.